# Coffee price by time of day: Holm's method

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Two-Way Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A campus coffee cart tried three prices for a regular coffee (\$2.50, \$3.50, \$4.50) at four times of day. Each combination was tried on three days, and cups sold were counted.

**The question.** Does the effect of price depend on the time of day?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/two_way_anova.csv` (36 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Price,Time of day,Cups sold
$2.50,Morning,43.0
$2.50,Morning,39.0
$2.50,Morning,41.0
$2.50,Lunch,32.0
$2.50,Lunch,35.0
$2.50,Lunch,34.0
$2.50,Afternoon,21.0
$2.50,Afternoon,27.0
$2.50,Afternoon,27.0
$2.50,Evening,15.0
$2.50,Evening,13.0
$2.50,Evening,14.0
$3.50,Morning,29.0
$3.50,Morning,31.0
$3.50,Morning,30.0
$3.50,Lunch,27.0
$3.50,Lunch,23.0
$3.50,Lunch,20.0
$3.50,Afternoon,22.0
$3.50,Afternoon,26.0
$3.50,Afternoon,26.0
$3.50,Evening,12.0
$3.50,Evening,11.0
$3.50,Evening,12.0
$4.50,Morning,26.0
$4.50,Morning,28.0
$4.50,Morning,25.0
$4.50,Lunch,22.0
$4.50,Lunch,23.0
$4.50,Lunch,24.0
$4.50,Afternoon,20.0
$4.50,Afternoon,20.0
$4.50,Afternoon,19.0
$4.50,Evening,10.0
$4.50,Evening,7.0
$4.50,Evening,6.0

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per observation: the response and the two factors. Rows with a blank are
# dropped. Levels keep the order they first appear in; every combination of levels
# needs data, and some cell needs at least two observations.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Cups sold"
column_factor = "Price"
row_factor = "Time of day"
frame = pd.DataFrame({"y": pd.to_numeric(data[response], errors="coerce"),
                      "A": data[column_factor], "B": data[row_factor]}).dropna()
column_levels = list(frame["A"].unique())
row_levels = list(frame["B"].unique())
frame["A"] = pd.Categorical(frame["A"], categories=column_levels)
frame["B"] = pd.Categorical(frame["B"], categories=row_levels)

## Descriptive statistics

In [ ]:
# Each level of the column factor (A), of the row factor (B), then each cell.
def describe(by):
    table = frame.groupby(by, observed=True)["y"].agg(n="count", mean="mean", SD="std")
    table.index = [" * ".join(map(str, np.atleast_1d(key))) for key in table.index]
    table["SE"] = table["SD"] / np.sqrt(table["n"])
    return table.assign(**{"mean - SD": table["mean"] - table["SD"],
                           "mean + SD": table["mean"] + table["SD"],
                           "mean - SE": table["mean"] - table["SE"],
                           "mean + SE": table["mean"] + table["SE"]})[
        ["n", "mean", "SD", "mean - SD", "mean + SD", "SE", "mean - SE", "mean + SE"]]


descriptive = pd.concat([describe("A"), describe("B"), describe(["A", "B"])],
                        keys=["A", "B", "A * B"])
print(descriptive.to_string())

## Test for homogeneity of variance

In [ ]:
# Levene's original form across the cells, on deviations from each cell's mean
# (center="mean"; SciPy's default is the median).
cells = [group["y"].to_numpy() for _, group in frame.groupby(["A", "B"], observed=True)]
levene = stats.levene(*cells, center="mean")
homogeneity = pd.DataFrame([[levene.statistic, len(cells) - 1, len(frame) - len(cells),
                             levene.pvalue]], columns=["F", "df1", "df2", "P"])
print()
print(homogeneity.to_string(index=False))

## Analysis of variance table

In [ ]:
# The effects use sum-to-zero coding (C(..., Sum)). With equal cell sizes the three
# types of sums of squares coincide with the textbook decomposition; with unequal
# cells they differ in what each main effect is adjusted for. The interaction is
# entered last in every type, so its line is the same in all three.
fit = smf.ols("y ~ C(A, Sum) * C(B, Sum)", data=frame).fit()
# Type III: each effect measured after the other two, so the table depends neither
# on the order of the factors nor on which level is listed last.
table = anova_lm(fit, typ=3).drop(index="Intercept")
ss_error, df_error = table.loc["Residual", "sum_sq"], table.loc["Residual", "df"]
ms_error = ss_error / df_error
anova = pd.DataFrame({"SS": table["sum_sq"].to_numpy(), "df": table["df"].to_numpy()},
                     index=["A (column)", "B (row)", "A * B", "Error"])
anova["MS"] = anova["SS"] / anova["df"]
anova["F"] = anova["MS"] / ms_error
anova["P"] = stats.f.sf(anova["F"], anova["df"], df_error)
anova.loc["Error", ["F", "P"]] = np.nan
anova.loc["Total"] = [((frame["y"] - frame["y"].mean()) ** 2).sum(), len(frame) - 1,
                      np.nan, np.nan, np.nan]
print()
print(anova)

columns = ["level 1", "level 2", "mean 1", "mean 2", "difference", "SE", "statistic", "P"]


# One family of pairwise comparisons: every pair of the given levels, statistic =
# |difference| / SE with SE = sqrt(MS error x (1/n1 + 1/n2)), on the error df of the
# whole model. The family is what an adjustment counts -- one factor's levels, or the
# levels of one factor inside one level of the other.
def pair_rows(values, levels):
    rows = []
    for a, b in combinations(levels, 2):
        x, z = values[a], values[b]
        se = np.sqrt(ms_error * (1 / len(x) + 1 / len(z)))
        difference = abs(x.mean() - z.mean())
        rows.append([a, b, x.mean(), z.mean(), difference, se, difference / se])
    t, k = np.array([row[-1] for row in rows]), len(levels)
    # Holm: Bonferroni step-down -- the smallest P value times m, the next times m - 1, ...
    p = multipletests(2 * stats.t.sf(t, df_error), method="holm")[1]
    return [[*row, p_value] for row, p_value in zip(rows, p)]

## Multiple comparison

In [ ]:
# Every pair of levels of each factor, on the plain level means and the pooled error;
# each factor is a family of its own.
pairs = pd.DataFrame(
    pair_rows({a: g["y"].to_numpy() for a, g in frame.groupby("A", observed=True)}, column_levels)
    + pair_rows({b: g["y"].to_numpy() for b, g in frame.groupby("B", observed=True)}, row_levels),
    columns=columns)
print()
print(pairs.to_string(index=False))

## Least-squares means

In [ ]:
# The mean each level would have if every cell weighed the same: the average of its
# cell means, with SE = sqrt(MS error x sum(1 / n of each cell)) / number of cells.
# With equal cell sizes this is the plain mean.
confidence = 95 / 100
critical = stats.t.ppf(0.5 + confidence / 2, df_error)
cell_mean = frame.pivot_table("y", "A", "B", aggfunc="mean", observed=True)
cell_n = frame.pivot_table("y", "A", "B", aggfunc="count", observed=True)
rows = []
for c in column_levels:
    se = np.sqrt(ms_error * (1 / cell_n.loc[c]).sum()) / len(row_levels)
    rows.append([c, cell_n.loc[c].sum(), cell_mean.loc[c].mean(), se])
for r in row_levels:
    se = np.sqrt(ms_error * (1 / cell_n[r]).sum()) / len(column_levels)
    rows.append([r, cell_n[r].sum(), cell_mean[r].mean(), se])
for c in column_levels:
    for r in row_levels:
        rows.append([f"{c} * {r}", cell_n.loc[c, r], cell_mean.loc[c, r],
                     np.sqrt(ms_error / cell_n.loc[c, r])])
lsmeans = pd.DataFrame(rows, columns=["level", "n", "LS mean", "SE"])
lsmeans["lower"] = lsmeans["LS mean"] - critical * lsmeans["SE"]
lsmeans["upper"] = lsmeans["LS mean"] + critical * lsmeans["SE"]
print()
print(lsmeans.to_string(index=False))

## Simple main effects

In [ ]:
# One factor tested separately at each level of the other. The error is the pooled
# one from the whole model -- not each slice's own -- so it keeps all its degrees of
# freedom. First the column factor within each row level, then the reverse.
simple = {}
for outer, inner, means, counts in (("B", "A", cell_mean.T, cell_n.T),
                                    ("A", "B", cell_mean, cell_n)):
    tests, pairs_within = [], []
    for level in means.index:
        m, k = means.loc[level], counts.loc[level]
        ss = (k * (m - (k * m).sum() / k.sum()) ** 2).sum()
        df = len(m) - 1
        f_value = (ss / df) / ms_error
        tests += [[level, inner, ss, df, ss / df, f_value, stats.f.sf(f_value, df, df_error)],
                  [level, "Error", ss_error, df_error, ms_error, np.nan, np.nan]]
        # The pairs inside this one level are a family of their own.
        values = {i: g["y"].to_numpy()
                  for i, g in frame[frame[outer] == level].groupby(inner, observed=True)}
        pairs_within += [[level, *row] for row in pair_rows(values, list(m.index))]
    simple[outer] = {
        "means": means, "n": counts,
        "replications": pd.DataFrame([[counts.to_numpy().min(), counts.to_numpy().max()]],
                                     columns=["min", "max"]),
        "tests": pd.DataFrame(tests, columns=["level", "source", "SS", "df", "MS", "F", "P"]),
        "pairs": pd.DataFrame(pairs_within, columns=["level", *columns]),
    }
    print(f"\n{inner} within each level of {outer}: means, n, and the tests")
    print(means)
    print(counts)
    print(simple[outer]["tests"].to_string(index=False))
    print(simple[outer]["pairs"].to_string(index=False))

## What the result says

The interaction between price and time of day has F = 6.77 and a P value far below 0.01. The P value is the chance of an effect at least this large if there were none. So the effect of price depends on the time of day, and prices are compared inside each time.

Holm's method corrects the P values in steps: the smallest gets the full correction, later ones a lighter one. So it finds a little more than Tukey's test. The step from \$3.50 to \$4.50 now counts as significant in the morning (P = 0.031) and in the evening (P = 0.040). In the afternoon, \$2.50 against \$3.50 stays far from significant, P = 0.837.

The decision is the same: \$3.50 costs almost no customers in the afternoon.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.